Dr. Xiao Li xiaoli@pku.edu.cn, 2026 Summer 

**AI for Cell Image Analysis** <a id=0> </a>

# <span style="font-size:30px"> Part 2. Introduction to Deep Learning</span>

This notebook is designed for students with zero background in deep learning, progressively introducing core **deep learning** concepts through five dataset-network combinations:
- entry-level: handwritten digit recognition using MLP + MNIST
- advancing to: blood-cell image classification using CNN + BloodMNIST
---

## <span style="font-size:25px"> Task 0: Environment Setup </span>

Installing necessary Python libraries

In [ ]:
# Install PyTorch and other dependencies for image processing
%pip install torch torchvision torchaudio
# If you have GPU support, you can try the GPU version by uncommenting the next line.
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
%pip install tqdm medmnist


---

## <span style="font-size:25px"> Task 1: MNIST Classfication </span>

`MNIST` (Modified National Institute of Standards and Technology) is a handwritten digit recognition dataset, widely regarded as the "Hello World" of computer vision. The dataset contains 70,000 grayscale images of 28×28 pixels, with 60,000 for training and 10,000 for testing. Each image represents a single handwritten digit from 0 to 9. MNIST has become a classic introductory dataset for deep learning due to its simplicity and standardization.

### 1.1 Imports

This part import required modules.

In [ ]:
#pip install --upgrade typing-extensions

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm #tqdm is a Python library for progress bar

This part automatically detect if you have GPU available. The program will use  GPU if available, otherwise will use CPU.

In [ ]:
# Automatic device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)

### 1.2 Load Data

In [ ]:
import PIL
from PIL import Image

In [ ]:
# Load the raw MNIST dataset
mnist_train_raw = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=None)
mnist_test_raw = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=None)

We use the _root_ parameter to define where to save the data. "./" stands for the current working directory of this notebook. You could open your fold containing this notebook to see a new folder called "mnistdata" is created: ./data/MNIST/raw <br>

The _train_ parameter is set to True because we are initializing the MNIST training dataset, and is set to False if we are saving the testing dataset. <br>

The _download_ parameter is set to True because we want to download it if it’s not already present in our data folder. <br>

The _transform_ parameter is set to None because we don’t want to apply any image manipulation transforms at this time. <br>

We can see that each element in the dataset contains a tuple. A tuple is like a list with fixed value. THe tuple contains a PIL.image object and a target lable（0~9).

**Explore the Data**🔍 <br>

In [ ]:
#explore the dataset
print(f"the data type of mnist_train_raw is: {type(mnist_train_raw)}")
print(f"Training set size: {len(mnist_train_raw)}")
print(f"Testing set size: {len(mnist_test_raw)}")

#TODO set to a certain index to view the data
n = 
print(f"the type nth element: {type(mnist_train_raw[n])}") 
print(f"the nth element in the raw training set: {mnist_train_raw[n]}") 

In [ ]:
#save the PIL image object and corresponding lable 
#for nth element in two seperate variables
image_n,label_n = mnist_train_raw[n]
print(f"the image format, size and mode for nth image is: {image_n.format},{image_n.size},{image_n.mode}") 
print(f"the label for nth image is: {label_n}") 
image_n #show image_n

PIL.image format can be 'PNG, 'JPEG' etc. "None" format is default for a PIL.image object. image.mode 'L' means grayscale. 

In [ ]:
# convert an image object to an numpy array
pixels = np.array(image_n)  

# show the array as image
plt.imshow(pixels,cmap="gray")
plt.title(f"target: {label_n}")
plt.show()
#
print(f"the size of pixel: {pixels.shape}")  #total number of pixels = 28x28
pixels

In [ ]:
# get pixel data (as Tuples) from PIL.image object
# TODO set pixel coordinate: x=column, y=row  
x = 
y = 
pixel = image_n.getpixel((x, y)) 
print(pixel)  # 0 for black and 225 for white 


To train a neural network model, the data for the images need to be standardized/normalized and store in tensors. 

📘**Tensor** is a generalization of scalars, vectors, and matrices in mathematics.

| Tensor Order | Example            | Shape       | Description         |
| ------------ | ------------------ | ----------- | ------------------- |
| 0D           | `3.14`             | `()`        | Scalar              |
| 1D           | `[1, 2, 3]`        | `(3,)`      | Vector              |
| 2D           | `[[1, 2], [3, 4]]` | `(2, 2)`    | Matrix              |
| 3D+          | `[[[...]]]`        | `(n, m, k)` | Tensor (3D or more) |


In machine learning, especially in image processing, tensors are used to represent data like images, batches of images, and so on. 

The terms tensor and NumPy array are similar, but they  have different capabilities, especially in deep learning frameworks like TensorFlow or PyTorch. A tensor is a more powerful and flexible structure than a plain NumPy array, especially when training models on GPUs or using automatic differentiation. All tensors can be thought of as n-dimensional arrays, but not all n-dimensional arrays (like np.array) are tensors in the deep learning sense.

Pytorch can convert PIL.image object to tensor through <a style = 'color:purple'> transform.ToTensor() </a> setting. 


In [ ]:
#transform MNIST raw data to tensor
mnist_train_tensor = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())

# Get all the image pixel data
loader = DataLoader(mnist_train_tensor, batch_size=60000, shuffle=False)
images, _ = next(iter(loader))  # images shape: [60000, 1, 28, 28]


In [ ]:
print(images.shape)
#N=60000 number of images
#C=1, number of channels, grayscale only 1 channel
#H=28, height, number of rows
#W=28, width, number of columns

In [ ]:
# Flatten to [N, C*H*W]
#images = images.view(images.size(0), -1)  # shape: [60000, 784]
images = images.reshape(images.size(0), -1)  # shape: [60000, 784]
#print(images.shape)

# Compute mean and std
mean = images.mean()
std = images.std()

print(f"Mean: {mean:.4f}")
print(f"Std: {std:.4f}")

The raw data ontains pixel grayscale values ranging from 0 to 225 and need to be normalized to the range `[0, 1]`

- Standardizes the pixel values using the formula:

    $ \text{normalized value} = \frac{\text{pixel} - \mu}{\sigma} $

- In this case: $ \mu $ = 0.1307, $ \sigma $ = 0.3081. These are the precomputed mean and standard deviation of the MNIST dataset.


In [ ]:
# Data preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST mean and std
])

# Load normalized data as tensors
mnist_train = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = torchvision.datasets.MNIST(root='./data', train=False, transform=transform)


In [ ]:
# Create data loaders
# Load parts of the data (32 iamges)
train_loader = DataLoader(mnist_train, batch_size=32, shuffle=True)
test_loader = DataLoader(mnist_test, batch_size=32, shuffle=False)

# Visualize data samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i in range(10):
    row = i // 5
    col = i % 5
    image, label = mnist_train[i]
    axes[row, col].imshow(image.squeeze(), cmap='gray', interpolation='none')
    axes[row, col].set_title(f'Label: {label}')
    axes[row, col].axis('off')

plt.suptitle('MNIST Sample Visualization')
plt.tight_layout()
plt.show()

In [ ]:
print(f"the nth element in the raw training set: {mnist_train_raw[n]}") 
print(f"the nth element in the transformed training set:\n {mnist_train[n]}") 

In [ ]:
# Display dataset information
print("\nMNIST Dataset Information:")
print(f"Image shape: {mnist_train[n][0].shape}")
print(f"Number of classes: 10 (digits 0-9)")

https://www.datascienceweekly.org/tutorials/pytorch-mnist-load-mnist-dataset-from-pytorch-torchvision 

### 1.3 Define MLP Model

`Multilayer Perceptron (MLP)` is the most basic feedforward neural network architecture, consisting of an **input layer**, one or more **hidden layers**, and an **output layer**. Each layer is fully connected to the next layer, hence also called **fully connected (FC)** networks or dense networks. Each neuron in an MLP uses a _nonlinear activation function_, enabling the network to learn complex nonlinear mappings.

<img src="./img/MLP_mnist.png" alt="sigmoid function" style="width: 600px"/> </br>
**Figure 1**. An example multilayer perceptron (MLP) archetecture for MNIST handwritten digits classification.

Figure 1 illustrates a typical MLP archetecture. Let's go through the codes used for building a simple MLP.

 for the MNIST handwritten digits classification problem, the number of features for the input and output layers are fixed.

- _<font color='purple'>input layer</font>_</br>
    Number of input features = 28x28 = 784 </br> 
    <font face="Courier New"> nn.Flatten()</font> can be used to convert 2D images (28x28 pixels) into 1D vectors (784 elements). 
- _<font color='purple'>output layer</font>_</br>
    Number of output features = 10 </br>
    The MLP should outputs the classification results which fall into 10 categories of digits: 0, 1, 2, 3, 4, 5, 6, 7, 8, 9. 
- _<font color='purple'>Hidden layers</font>_ </br>
    MLP should have at least one hidden layer. The archetecture shown in figure 1 has two hidden layers. <font face="Courier New">nn.Dropout(<font color='blue'>p</font>) </font> helps prevent overfitting to the training data. </br>
    To define a MLP as shown, we can use codes like below:</br>
    <font face="Courier New">
        self.flatten = nn.Flatten() </br>
        self.fc1 = nn.Linear(784, <font color='blue'>X1</font>) </br>
        self.fc2 = nn.Linear(128, <font color='blue'>X2</font>) </br>
        self.fc3 = nn.Linear(<font color='blue'>X2</font>, 10) </br>
        self.dropout = nn.Dropout(<font color='blue'>p</font>) </br>
    </font>

After each fully connected layer (e.g., fc1) a nonlinear function should be introduced after each hidden layer to activate the neural network. Common nonlinear functions are `ReLu` and `sigmoid` function. </br>
Example code is </br>
<font face="Courier New">x = torch.relu(self.fc1(x)) </font>


**TODO**: Define your own MLP (SimpleMLP) by specifying X1，X2 (X3 ...) and p. </br>
    Note: MNIST dataset is a relatively simple and robust dataset, and the classification results are not very sensitive to the network archetecture. You can try to have one layer, two layers, three or even more layers and see the results. A recommended p value is 0.2, meaning dropping out 20% of features after each fc. Dropout is not necessary in this problem though. 


In [ ]:
#Defines neural network class called SimpleMLP that inherits from PyTorch's base nn.Module class
class SimpleMLP(nn.Module): 
    def __init__(self):
        super(SimpleMLP, self).__init__()
        # TODO: Define the network layers
        # Hint: You need Flatten, Linear layers, and Dropout


        # TODO end
        
    def forward(self, x):
        # TODO: Define the forward pass
        # Hint: Apply flatten, then linear layers with ReLU and dropout


        # TODO end

In [ ]:
# Create model instance
model = SimpleMLP()
print("MLP Model Structure:")
print(model)

# Calculate total parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal number of parameters: {total_params:,}")

The exact parameter count for this SimpleMLP is calculated as:

fc1 layer: 784×128 + 128 = 100,480 <br>
fc2 layer: 128×64 + 64 = 8,256 <br>
fc3 layer: 64×10 + 10 = 650 <br>

Total parameters = 100,480 + 8,256 + 650 = 109,386

Simple MLP has a common pattern in the network architecture:

**input layer → hidden layer(+non-linear activaton) → dropout → ...→ hidden layer(+non-linear activaton) → dropout → output layer**


### 1.4 Define Training and Evaluation Functions

In [ ]:
#Defines a function to train the model for one epoch
def train_one_epoch(model, train_loader, criterion, optimizer, epoch):
    model.train() #set the model to training mode (enables dropout, batch norm updates)
    #initializes counters: 
    total_loss = 0 #cumulative loss
    correct = 0 #correct predictions
    total = 0 #total samples processed
    
    #creates a progress bar with description showing current epoch
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1} [Training]', leave=False)
    
    for data, target in pbar:
        #  Move data to device for GPU support
        # Hint: Use .to(device) for both data and target
        data, target = data.to(device), target.to(device)
        # end
        
        # Implement the training step
        # Hint: zero_grad, forward pass, loss calculation, backward, optimizer step
        optimizer.zero_grad() #clears old gradients from previous batch
        output = model(data) #forward pass: computes model predictions
        loss = criterion(output, target) #calculates loss between predictions and true labels
        loss.backward() #backward pass: computes gradients
        optimizer.step() #updates model parameters using optimizer
        # end
        
        # Calculate statistics
        total_loss += loss.item()
        _, predicted = torch.max(output.data, 1) #gets predicted class (index with highest score)
        total += target.size(0) #counts total samples in batch
        correct += (predicted == target).sum().item() #counts correct predictions in batch
        
        # Update progress bar with current batch loss and accuracy
        current_acc = 100 * correct / total
        pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{current_acc:.2f}%'})
    
    #Calculates epoch average loss and accuracy
    avg_loss = total_loss / len(train_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy

In [ ]:
#defines evaluation function with similar parameters as training
def evaluate(model, test_loader, criterion, epoch):
    model.eval() #sets model to evaluation mode (disables dropout, fixes batch norm)
    #initializes same counters as training
    total_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(test_loader, desc=f'Epoch {epoch+1} [Validation]', leave=False)
    
    with torch.no_grad():
        for data, target in pbar:
            # Move data to device for GPU support
            data, target = data.to(device), target.to(device)
            # end
            
            # Implement the evaluation step
            # Hint: forward pass and loss calculation (no gradients needed)
            output = model(data)
            loss = criterion(output, target)
            #  end
            
            total_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            total += target.size(0)
            correct += (predicted == target).sum().item()
            
            # Update progress bar
            current_acc = 100 * correct / total
            pbar.set_postfix({'Loss': f'{loss.item():.4f}', 'Acc': f'{current_acc:.2f}%'})

    avg_loss = total_loss / len(test_loader)
    accuracy = 100 * correct / total
    return avg_loss, accuracy
    

### 1.5 Train the Model

In [ ]:
#  Move model to device for GPU support
# Hint: Use model.to(device)
model = model.to(device)
# end

# TODO: Define loss function and optimizer
# Hint: Use CrossEntropyLoss and Adam optimizer
criterion = 
optimizer = optim.Adam(model.parameters(), lr=0.001)
# TODO end

# Record training progress
train_losses = []
train_accuracies = []
test_losses = []
test_accuracies = []

epochs = 5
print("Starting training...")
print(f"Model will train on: {device}")

# Add main training loop with epoch progress
# Hint: Use tqdm for epoch progress and time tracking
start_time = time.time()

for epoch in tqdm(range(epochs), desc="Overall Progress"):
    epoch_start = time.time()
    
    # Train for one epoch
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, epoch)
    test_loss, test_acc = evaluate(model, test_loader, criterion, epoch)
    
    # Record metrics
    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    test_losses.append(test_loss)
    test_accuracies.append(test_acc)
    
    epoch_time = time.time() - epoch_start
    
    print(f'Epoch [{epoch+1}/{epochs}] - Time: {epoch_time:.1f}s')
    print(f'  Training   - Loss: {train_loss:.4f}, Accuracy: {train_acc:.2f}%')
    print(f'  Validation - Loss: {test_loss:.4f}, Accuracy: {test_acc:.2f}%')
    print('-' * 50)

total_time = time.time() - start_time
print(f"Training completed on {device}! Total time: {total_time:.1f}s")

### 1.6 Visualize Training Results

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot loss curves
ax1.plot(range(1, epochs+1), train_losses, 'b-', label='Training Loss', marker='o')
ax1.plot(range(1, epochs+1), test_losses, 'r-', label='Validation Loss', marker='s')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss During Training')
ax1.legend()
ax1.grid(True)

# Plot accuracy curves
# Plot the accuracy curves
# Hint: Similar to loss curves but for accuracies
ax2.plot(range(1, epochs+1), train_accuracies, 'b-', label='Training Accuracy', marker='o')
ax2.plot(range(1, epochs+1), test_accuracies, 'r-', label='Validation Accuracy', marker='s')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Accuracy During Training')
ax2.legend()
ax2.grid(True)


plt.tight_layout()
plt.show()

print(f"Final test accuracy: {test_accuracies[-1]:.2f}%")

### 1.7 Visualize Predictions

In [ ]:
def visualize_predictions(model, test_loader, num_samples=20):
    model.eval()
    
    # Get one batch of data
    data_iter = iter(test_loader)
    images, labels = next(data_iter)
    
    # Move data to device and get predictions
    # Hint: Move images to device, get outputs, then move back to CPU for visualization
    images_gpu = images.to(device)
    with torch.no_grad():
        outputs = model(images_gpu)
        _, predictions = torch.max(outputs, 1)
    
    # Move back to CPU for visualization
    images = images.cpu()
    predictions = predictions.cpu()
  
    
    # Visualize the prediction results
    # Hint: Create subplots and show images with true vs predicted labels
    fig, axes = plt.subplots(4, 5, figsize=(15, 12))
    axes = axes.ravel()
    
    for i in range(num_samples):
        img = images[i].squeeze()
        true_label = labels[i].item()
        pred_label = predictions[i].item()
        
        axes[i].imshow(img, cmap='gray')
        color = 'green' if true_label == pred_label else 'red'
        axes[i].set_title(f'True: {true_label}, Pred: {pred_label}', color=color)
        axes[i].axis('off')
    
    plt.suptitle('MNIST Test Results (Green=Correct, Red=Incorrect)', fontsize=16)
    plt.tight_layout()
    plt.show()
    
    # Calculate accuracy for displayed samples
    correct = (predictions[:num_samples] == labels[:num_samples]).sum().item()
    accuracy = correct / num_samples * 100
    print(f"Accuracy for displayed samples: {accuracy:.1f}% ({correct}/{num_samples})")

In [ ]:
# Visualize prediction results
visualize_predictions(model, test_loader)


While MLPs perform well on tabular data, they have been largely replaced by CNNs in image processing tasks due to their large parameter count and inability to effectively capture spatial features. We will learn CNNs in Task2.

---

## Task 2: BloodMNIST Classification with CNN

**BloodMNIST** is a biomedical microscopy dataset from the MedMNIST collection. It contains **17,092 RGB images** of individual normal blood cells grouped into **8 classes**. The official split contains 11,959 training images, 1,712 validation images, and 3,421 test images. Source images were center-cropped and resized to 28 × 28 pixels for a lightweight multi-class classification benchmark.

This task replaces everyday-object classification with a chemically and biologically relevant problem: learning cell morphology from stained blood-smear images. A CNN can learn local patterns such as cell boundaries, nuclear shape, granularity, and color distribution.

> **Important:** This notebook is an educational image-classification exercise, not a clinical diagnostic system.

References: [MedMNIST](https://medmnist.com/) · [MedMNIST API](https://github.com/MedMNIST/MedMNIST) · [BloodMNIST CNN project](https://github.com/Nilsvanesostos/BloodMNIST)


### 2.1 Imports

In [ ]:
import time
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm
from sklearn.metrics import classification_report, confusion_matrix

import medmnist
from medmnist import INFO

# Automatic device detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

torch.manual_seed(42)
np.random.seed(42)


### 2.2 Load Data

The MedMNIST API downloads the official dataset and preserves its predefined training, validation, and test splits. BloodMNIST images are RGB, so each tensor has shape **3 × 28 × 28**.

We normalize every channel from the range [0, 1] to approximately [−1, 1]. The test set is kept untouched until the final evaluation.

In [ ]:
# Dataset metadata
data_flag = 'bloodmnist'
info = INFO[data_flag]
DataClass = getattr(medmnist, info['python_class'])

class_names = [info['label'][str(i)] for i in range(len(info['label']))]
# Use a shorter display name for plots and prediction titles
class_names[3] = 'granulocytes'
n_channels = info['n_channels']
n_classes = len(class_names)

transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * n_channels, std=[0.5] * n_channels),
])

transform_eval = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5] * n_channels, std=[0.5] * n_channels),
])

# Store the downloaded dataset inside this project's data folder
data_root = './data'

print(f"Downloading/loading BloodMNIST from {data_root}...")
train_dataset = DataClass(split='train', root=data_root, transform=transform_train, download=True)
val_dataset = DataClass(split='val', root=data_root, transform=transform_eval, download=True)
test_dataset = DataClass(split='test', root=data_root, transform=transform_eval, download=True)

print("BloodMNIST has been downloaded and loaded successfully.")

batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Task: {info['task']}")
print(f"Channels: {n_channels}; classes: {n_classes}")
print(f"Train/validation/test: {len(train_dataset):,} / {len(val_dataset):,} / {len(test_dataset):,}")


In [ ]:
print("BloodMNIST classes:")
for index, name in enumerate(class_names):
    print(f"  {index}: {name}")

image, label = train_dataset[0]
print(f"One image tensor shape: {image.shape}")
print(f"One label shape: {np.asarray(label).shape}; value: {int(np.asarray(label).squeeze())}")


In [ ]:
# Undo normalization for display
def denormalize_blood_image(tensor):
    return torch.clamp(tensor * 0.5 + 0.5, 0, 1)

image_show = denormalize_blood_image(image)
plt.figure(figsize=(3, 3))
plt.imshow(image_show.permute(1, 2, 0))
plt.title(class_names[int(np.asarray(label).squeeze())])
plt.axis('off')
plt.show()


In [ ]:
# Visualize one BloodMNIST sample from each class
# Hint: Search the training set until all 8 labels have been found.
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
found = set()

for image, label in train_dataset:
    class_id = int(np.asarray(label).squeeze())
    if class_id not in found:
        ax = axes.ravel()[class_id]
        ax.imshow(denormalize_blood_image(image).permute(1, 2, 0))
        ax.set_title(class_names[class_id], fontsize=9)
        ax.axis('off')
        found.add(class_id)
    if len(found) == n_classes:
        break

plt.suptitle('BloodMNIST: One Training Image per Cell Type')
plt.tight_layout()
plt.show()



In [ ]:
# Inspect class balance in the training split
train_labels = np.asarray(train_dataset.labels).squeeze().astype(int)
class_counts = np.bincount(train_labels, minlength=n_classes)

plt.figure(figsize=(11, 4))
plt.bar(range(n_classes), class_counts, color='#7b68b5')
plt.xticks(range(n_classes), class_names, rotation=35, ha='right')
plt.ylabel('Number of training images')
plt.title('BloodMNIST Training-Class Distribution')
plt.tight_layout()
plt.show()

for name, count in zip(class_names, class_counts):
    print(f"{name:55s} {count:5d}")


### 2.3 Define CNN Model

A **Convolutional Neural Network (CNN)** learns local visual patterns by sliding trainable kernels across an image. Early layers often respond to edges and color transitions; deeper layers combine these signals into morphology-related features.

<img src="./img/CNN.png" alt="CNN architecture" style="width: 800px"/> </br>
**Figure 2.** A typical CNN architecture for image recognition.

**TODO:** Design the following CNN model for BloodMNIST:

- **Input layer:** 3 × 28 × 28 RGB image
- **Output layer:** 8 logits, one for each blood-cell class
- **Convolutional layers:** learn local color, texture, boundary, and nuclear patterns
- **Pooling layers:** reduce spatial size and computation
- **Fully connected layer:** converts learned features into class scores

Examine how the channel count changes from 3 → 32 → 64 → 128. Why does the spatial size decrease while the number of feature maps increases?
You could also define your own model.

In [ ]:

class BloodCellCNN(nn.Module):
    def __init__(self, num_classes=8):
        super().__init__()
        # TODO: Define convolutional layers
        self.features = nn.Sequential(
            nn.Conv2d(  ,  , kernel_size= , padding=1),
            nn.BatchNorm2d( ),
            nn.ReLU(inplace=True),
            nn.MaxPool2d( ),                      

            nn.Conv2d(  ,  , kernel_size=  , padding=1),
            nn.BatchNorm2d( ),
            nn.ReLU(inplace=True),
            nn.MaxPool2d( ),                     

            nn.Conv2d( ,  , kernel_size=3, padding=1),
            nn.BatchNorm2d( ),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((3, 3)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.30),
            nn.Linear(128 * 3 * 3, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.30),
            nn.Linear(128, num_classes),
        )
   # TODO end
    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

model = BloodCellCNN(num_classes=n_classes)
print("Blood-cell CNN structure:")
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# Check the tensor shapes before training
sample_images, sample_labels = next(iter(train_loader))
with torch.no_grad():
    sample_logits = model(sample_images)

print(f"Input batch:  {sample_images.shape}")
print(f"Labels:       {sample_labels.shape}")
print(f"Output logits:{sample_logits.shape}")
assert sample_logits.shape == (sample_images.size(0), n_classes)


### 2.4 Define Training and Evaluation Functions

`CrossEntropyLoss` is appropriate because BloodMNIST is a single-label, eight-class classification task. MedMNIST stores labels with shape `(batch, 1)`, so the functions flatten them to `(batch,)` before calculating the loss.

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, description='Training'):
    is_training = optimizer is not None
    model.train(is_training)
    total_loss, correct, total = 0.0, 0, 0

    progress = tqdm(loader, desc=description, leave=False)
    context = torch.enable_grad() if is_training else torch.no_grad()

    with context:
        for images, labels in progress:
            images = images.to(device)
            labels = labels.squeeze(1).long().to(device)

            if is_training:
                optimizer.zero_grad()

            logits = model(images)
            loss = criterion(logits, labels)

            if is_training:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
            progress.set_postfix(loss=f'{loss.item():.4f}', acc=f'{100 * correct / total:.2f}%')

    return total_loss / total, 100 * correct / total


### 2.5 Train the Model

The validation split guides model selection; it is not merged with the test set. The best validation checkpoint is restored before the final test evaluation.

In [ ]:
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 10
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []
best_val_accuracy = -1.0
best_state = None

print(f"Starting BloodMNIST CNN training on {device}...")
start_time = time.time()

for epoch in tqdm(range(epochs), desc='CNN Training Progress'):
    train_loss, train_acc = run_epoch(
        model, train_loader, criterion, optimizer,
        description=f'Epoch {epoch + 1} [Training]'
    )
    val_loss, val_acc = run_epoch(
        model, val_loader, criterion,
        description=f'Epoch {epoch + 1} [Validation]'
    )

    train_losses.append(train_loss)
    train_accuracies.append(train_acc)
    val_losses.append(val_loss)
    val_accuracies.append(val_acc)

    if val_acc > best_val_accuracy:
        best_val_accuracy = val_acc
        best_state = {name: tensor.detach().cpu().clone()
                      for name, tensor in model.state_dict().items()}

    print(f"Epoch [{epoch + 1}/{epochs}] | "
          f"train loss {train_loss:.4f}, acc {train_acc:.2f}% | "
          f"val loss {val_loss:.4f}, acc {val_acc:.2f}%")

print(f"Training time: {time.time() - start_time:.1f} s")
print(f"Best validation accuracy: {best_val_accuracy:.2f}%")

model.load_state_dict(best_state)
test_loss, test_accuracy = run_epoch(model, test_loader, criterion, description='Final Test')
print(f"Final held-out test loss: {test_loss:.4f}")
print(f"Final held-out test accuracy: {test_accuracy:.2f}%")


### 2.6 Visualize Training Results

In [ ]:
epoch_axis = range(1, epochs + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epoch_axis, train_losses, 'o-', label='Training loss')
ax1.plot(epoch_axis, val_losses, 's-', label='Validation loss')
ax1.set(xlabel='Epoch', ylabel='Loss', title='BloodMNIST CNN Loss')
ax1.grid(True, alpha=0.3)
ax1.legend()

ax2.plot(epoch_axis, train_accuracies, 'o-', label='Training accuracy')
ax2.plot(epoch_axis, val_accuracies, 's-', label='Validation accuracy')
ax2.set(xlabel='Epoch', ylabel='Accuracy (%)', title='BloodMNIST CNN Accuracy')
ax2.grid(True, alpha=0.3)
ax2.legend()

plt.tight_layout()
plt.show()


### 2.7 Visualize Predictions and Errors

Green titles indicate correct predictions and red titles indicate errors. Error analysis is especially important when classes have similar morphology or unequal sample counts.

In [ ]:
def collect_predictions(model, loader):
    model.eval()
    all_images, all_labels, all_predictions = [], [], []
    with torch.no_grad():
        for images, labels in loader:
            logits = model(images.to(device))
            predictions = logits.argmax(dim=1).cpu()
            all_images.append(images.cpu())
            all_labels.append(labels.squeeze(1).long().cpu())
            all_predictions.append(predictions)
    return (torch.cat(all_images), torch.cat(all_labels), torch.cat(all_predictions))

test_images, test_labels, test_predictions = collect_predictions(model, test_loader)

num_samples = 20
fig, axes = plt.subplots(4, 5, figsize=(15, 12))
for i, ax in enumerate(axes.ravel()):
    image = denormalize_blood_image(test_images[i])
    truth = test_labels[i].item()
    prediction = test_predictions[i].item()
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(f"True: {class_names[truth]}\nPred: {class_names[prediction]}",
                 color='green' if truth == prediction else 'red', fontsize=9)
    ax.axis('off')

plt.suptitle('BloodMNIST Test Predictions (Green = Correct, Red = Incorrect)')
plt.tight_layout()
plt.show()


In [ ]:
# Classification report and confusion matrix
blood_y_test = test_labels.numpy()
blood_y_pred = test_predictions.numpy()

print("Classification Report:")
print(classification_report(
    blood_y_test,
    blood_y_pred,
    labels=range(n_classes),
    target_names=class_names,
    digits=4,
    zero_division=0
))

blood_conf_mat = confusion_matrix(
    blood_y_test,
    blood_y_pred,
    labels=range(n_classes)
)
print("Confusion Matrix:")
print(blood_conf_mat)

# Accuracy for each cell type = correct predictions / samples of that type
class_totals = blood_conf_mat.sum(axis=1)
class_correct = np.diag(blood_conf_mat)
class_accuracies = np.divide(
    class_correct, class_totals,
    out=np.zeros_like(class_correct, dtype=float),
    where=class_totals != 0
)

print("\nAccuracy for Each Cell Type:")
for name, accuracy, correct, total in zip(
    class_names, class_accuracies, class_correct, class_totals
):
    print(f"{name:15s}: {accuracy * 100:6.2f}% ({correct}/{total})")

# Preserve the original row-normalized confusion-matrix heat map
confusion = torch.zeros(n_classes, n_classes, dtype=torch.int64)
for truth, prediction in zip(test_labels, test_predictions):
    confusion[truth, prediction] += 1

row_totals = confusion.sum(dim=1, keepdim=True).clamp_min(1)
confusion_percent = confusion.float() / row_totals * 100


In [ ]:

fig, ax = plt.subplots(figsize=(10, 8))
image = ax.imshow(confusion_percent.numpy(), cmap='Blues', vmin=0, vmax=100)
ax.set_xticks(range(n_classes), class_names, rotation=45, ha='right')
ax.set_yticks(range(n_classes), class_names)
ax.set_xlabel('Predicted class')
ax.set_ylabel('True class')
ax.set_title('BloodMNIST Test Confusion Matrix (Row %)')

for row in range(n_classes):
    for column in range(n_classes):
        value = confusion_percent[row, column].item()
        ax.text(column, row, f'{value:.0f}', ha='center', va='center',
                color='white' if value > 50 else 'black', fontsize=8)

fig.colorbar(image, ax=ax, label='Percentage of true class')
plt.tight_layout()
plt.show()

## Task 3: Compare CPU and GPU Performance

Choose one task from task 1 or 2 and run again on GPU on another notebook. Record the total training time.
- Plot the time needed for CPU and GPU configurations. </br>
- Plot a total cost vs machine configuration bar chart. (optional)

Recomended comparison: 
| Machine Type       | CPU      | CPU       | GPU          | GPU                     |
|--------------------|----------|-----------|--------------|-------------------------|
| Machine Configuration | c2m4     | c4m16     |c3m4* NVIDIA T4    | c8_m32_1 * NVIDIA V100 <br> (optional) |
| Cost (¥/h)         | 0        | 0.6    | 0.66         | 11                      |

In [ ]:
#TODO: Generate a barchart plot for training time comparison
#x-axis represents machine configuration (e.g., CPU(c2m4), GPU(NVIDIA T4)
#y-axis represents training time in seconds

#TODO: end

#TODO: Plot machine cost comparison (optional)
#x-axis represents machine type (e.g., CPU(c2m4), GPU(NVIDIA T4)
#y-axis represents total cost in yuan


#TODO: end